# Model Registry and Policy Outputs

## 1. Objective
Create a traceable registry of existing BharatCrop Intelligence model artifacts and define policy outputs that communicate their intended use, validation status, limitations, and operational restrictions.

## 2. Key Questions
- Which trained models and metadata artifacts currently exist?
- What prediction target and feature contract does each model use?
- What evidence supports each model's evaluation or readiness status?
- Which models are suitable for experimentation, integration testing, or further validation?
- What checks should be required before model predictions are used in downstream applications?
- How can model versions, artifact paths, and policy decisions be recorded reproducibly?

## 3. Methodology
- Discover existing model artifacts and associated metadata.
- Inspect artifact availability, model contracts, and recorded evaluation evidence.
- Build a registry from verified information rather than assumptions.
- Define policy rules for missing inputs, unsupported categories, validation gaps, and intended use.
- Export machine-readable registry and policy outputs.

## 4. Governance Principles
- Artifact availability does not establish model quality.
- Successful loading or prediction does not establish production readiness.
- Evaluation metrics must retain their evaluation context and limitations.
- Missing or unverified evidence must be recorded explicitly.
- Policy outputs must not claim validation, compliance, or deployment approval without supporting evidence.

## 5. Constraints
- Reuse existing models, metadata, evaluation reports, and datasets.
- Do not retrain models or create synthetic evidence of readiness.
- Preserve the distinction between verified facts, documented claims, and unresolved checks.
- Avoid overwriting existing model artifacts.
- Keep registry and policy outputs reproducible and auditable.

## 6. Expected Outputs
A model registry, a policy-rules artifact, a validation-status summary, and documentation explaining model eligibility, restrictions, and remaining checks.

In [1]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    r"C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence"
)

MODELS_DIR = PROJECT_ROOT / "models"

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}")

if not MODELS_DIR.exists():
    raise FileNotFoundError(f"Models directory not found: {MODELS_DIR}")

artifact_records = []

for path in sorted(MODELS_DIR.rglob("*")):
    if path.is_file():
        artifact_records.append({
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "filename": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": path.stat().st_size,
            "modified_time": pd.Timestamp(
                path.stat().st_mtime, unit="s"
            ).isoformat()
        })

artifact_inventory = pd.DataFrame(artifact_records)

print("Project root:", PROJECT_ROOT)
print("Models directory:", MODELS_DIR)
print("Total model-directory files:", len(artifact_inventory))

if artifact_inventory.empty:
    print("No files found under the models directory.")
else:
    display(artifact_inventory)

inventory_path = MODELS_DIR / "model_artifact_inventory.csv"
artifact_inventory.to_csv(inventory_path, index=False)

print("\nInventory saved:", inventory_path)


Project root: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
Models directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models
Total model-directory files: 23


,relative_path,filename,extension,size_bytes,modified_time
0,models\agricultural_risk_intelligence\agricult...,agricultural_risk_metadata.json,.json,1502,2026-11-01T14:43:47.406533003
1,models\agricultural_risk_intelligence\district...,district_crop_risk_summary.csv,.csv,1238057,2026-11-01T14:43:47.392401218
2,models\agricultural_risk_intelligence\extreme_...,extreme_yield_deviations_review.csv,.csv,60568,2026-11-01T14:43:47.404052734
3,models\agricultural_risk_intelligence\yield_ri...,yield_risk_indicators.csv,.csv,27450992,2026-11-01T14:43:47.293579340
4,models\crop_recommendation\crop_recommendation...,crop_recommendation_evaluation.json,.json,1835,2026-11-01T13:25:37.596370459
5,models\crop_recommendation\crop_recommendation...,crop_recommendation_metadata.json,.json,1262,2026-11-01T13:27:29.252828359
6,models\crop_recommendation\crop_recommendation...,crop_recommendations_projection_2025.csv,.csv,530998,2026-11-01T13:32:34.743075132
7,models\crop_recommendation\crop_recommendation...,crop_recommendations_ranked.csv,.csv,2909453,2026-11-01T13:24:11.505210638
8,models\model_explainability_and_scenarios\crop...,crop_season_error_analysis.csv,.csv,5103,2026-11-01T15:02:08.101760626
9,models\model_explainability_and_scenarios\crop...,crop_season_prediction_bias.csv,.csv,5604,2026-11-01T15:02:50.483734608



Inventory saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\model_artifact_inventory.csv


In [2]:

import json
from pathlib import Path
import pandas as pd

JSON_DIRS = [
    MODELS_DIR / "crop_recommendation",
    MODELS_DIR / "weather_intelligence",
    MODELS_DIR / "yield_prediction",
    MODELS_DIR / "agricultural_risk_intelligence",
]

json_records = []

for directory in JSON_DIRS:
    if not directory.exists():
        print("Directory not found:", directory)
        continue

    for path in sorted(directory.glob("*.json")):
        try:
            with open(path, "r", encoding="utf-8") as f:
                content = json.load(f)

            json_records.append({
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
                "filename": path.name,
                "top_level_keys": list(content.keys())
                    if isinstance(content, dict) else [],
                "content": content,
                "read_status": "OK"
            })

        except Exception as exc:
            json_records.append({
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
                "filename": path.name,
                "top_level_keys": [],
                "content": None,
                "read_status": f"ERROR: {exc}"
            })

metadata_report = pd.DataFrame(json_records)

print("JSON artifacts inspected:", len(metadata_report))
display(
    metadata_report[
        ["relative_path", "top_level_keys", "read_status"]
    ]
)

print("\nDetailed JSON contents:")
for record in json_records:
    print("\n" + "=" * 80)
    print(record["relative_path"])
    print("=" * 80)
    print(json.dumps(record["content"], indent=2, default=str))


JSON artifacts inspected: 7


,relative_path,top_level_keys,read_status
0,models\crop_recommendation\crop_recommendation...,"[evaluation_period, eligible_prediction_rows, ...",OK
1,models\crop_recommendation\crop_recommendation...,"[project, model_name, model_artifact, target, ...",OK
2,models\weather_intelligence\low_yield_model_ev...,"[model_task, target_definition, train_years, t...",OK
3,models\weather_intelligence\weather_intelligen...,"[module, notebook, source_dataset, period, dis...",OK
4,models\yield_prediction\individual_crop_hgb_lo...,"[model_name, project, candidate_scope, feature...",OK
5,models\yield_prediction\yield_model_deployment...,"[report_name, generated_at_utc, model_path, me...",OK
6,models\agricultural_risk_intelligence\agricult...,"[module, source_dataset, analysis_period, prep...",OK



Detailed JSON contents:

models\crop_recommendation\crop_recommendation_evaluation.json
{
  "evaluation_period": [
    2023,
    2024
  ],
  "eligible_prediction_rows": 8433,
  "evaluated_groups": 2365,
  "top1_hit_rate": 0.7919661733615222,
  "top3_hit_rate": 0.99830866807611,
  "mean_spearman_correlation": 0.7919884758726925,
  "by_year": [
    {
      "year": 2023,
      "groups": 1175,
      "top1_hit_rate": 0.788936170212766,
      "top3_hit_rate": 0.9982978723404256,
      "mean_spearman": 0.7820929849596775
    },
    {
      "year": 2024,
      "groups": 1190,
      "top1_hit_rate": 0.7949579831932773,
      "top3_hit_rate": 0.9983193277310924,
      "mean_spearman": 0.8017674513334093
    }
  ],
  "by_candidate_count": [
    {
      "Candidate_Bucket": "2 crops",
      "groups": 887,
      "top1_hit_rate": 0.8658399098083427,
      "top3_hit_rate": 1.0,
      "mean_spearman": 0.7313769751693002
    },
    {
      "Candidate_Bucket": "3 crops",
      "groups": 716,
      "top1

In [3]:

# Inspect likely governance fields without assuming every file
# follows the same metadata schema.

KEY_TERMS = (
    "model", "artifact", "target", "feature", "metric",
    "score", "mae", "rmse", "r2", "accuracy", "f1",
    "train", "test", "valid", "evaluation", "status",
    "ready", "verdict", "limit", "risk", "period",
    "threshold", "decision", "policy"
)

evidence_rows = []

for record in json_records:
    content = record["content"]

    row = {
        "relative_path": record["relative_path"],
        "read_status": record["read_status"]
    }

    if isinstance(content, dict):
        for key, value in content.items():
            key_lower = str(key).lower()

            if any(term in key_lower for term in KEY_TERMS):
                if isinstance(value, (dict, list)):
                    rendered = json.dumps(value, default=str)
                else:
                    rendered = str(value)

                row[key] = rendered[:500]

    evidence_rows.append(row)

evidence_df = pd.DataFrame(evidence_rows)

print("Evidence comparison:")
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", 65,
    "display.width", 220
):
    display(evidence_df)

evidence_path = MODELS_DIR / "model_evidence_comparison.csv"
evidence_df.to_csv(evidence_path, index=False)

print("\nEvidence comparison saved:", evidence_path)


Evidence comparison:


,relative_path,read_status,evaluation_period,model_name,model_artifact,target,evaluation_years,features,limitations,model_task,target_definition,train_years,test_years,train_rows,test_rows,train_event_rate,test_event_rate,metrics,deployment_status,period,categorical_features,numeric_features,training_rows,training_year_start,training_year_end,validation_windows,validation_metrics,model_path,verdict,analysis_period,risk_definition
0,models\crop_recommendation\crop_recommendation_evaluation.json,OK,"[2023, 2024]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,models\crop_recommendation\crop_recommendation_metadata.json,OK,NaN,HistGradientBoostingRegressor,models\yield_prediction\individual_crop_hgb_location.joblib,Yield,"[2023, 2024]","[""Crop"", ""Season"", ""Crop_Category"", ""State"", ""District"", ""FE_...","[""The saved ranking CSV contains retrospective rankings for 2...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,models\weather_intelligence\low_yield_model_evaluation.json,OK,NaN,NaN,NaN,NaN,NaN,NaN,"[""Annual district weather is aligned using a starting-calenda...",Experimental low-yield classification,Current yield below 80% of the median of the previous 2-3 ava...,"[2015, 2022]","[2023, 2024]",47309,12692,0.1447081950580228,0.12976678222502364,"{""constant_baseline"": {""roc_auc"": 0.5, ""pr_auc"": 0.1297667822...",NOT READY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,models\weather_intelligence\weather_intelligence_metadata.json,OK,NaN,NaN,NaN,NaN,NaN,NaN,"[""Weather uses starting-calendar-year proxy alignment."", ""Ful...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""start_year"": 2013, ""end_year"": 2024}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,models\yield_prediction\individual_crop_hgb_location_metadata...,OK,NaN,HistGradientBoostingRegressor,NaN,Yield,NaN,"[""Crop"", ""Season"", ""Crop_Category"", ""State"", ""District"", ""FE_...","[""Final holdout was repeatedly inspected during diagnostics.""...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[""Crop"", ""Season"", ""Crop_Category"", ""State"", ""District""]","[""FE_YearStart""]",35784,2013,2022,"[{""train_through"": 2018, ""validation_years"": [2019, 2020]}, {...","{""2019-2020"": {""MAE"": 448.306, ""RMSE"": 672.961, ""R2"": 0.798},...",NaN,NaN,NaN,NaN
5,models\yield_prediction\yield_model_deployment_readiness_repo...,OK,NaN,HistGradientBoostingRegressor,NaN,Yield,NaN,"[""Crop"", ""Season"", ""Crop_Category"", ""State"", ""District"", ""FE_...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\model...,CONDITIONALLY READY FOR INTEGRATION,NaN,NaN
6,models\agricultural_risk_intelligence\agricultural_risk_metad...,OK,NaN,NaN,NaN,NaN,NaN,NaN,"[""Exploratory historical indicators, not calibrated future-ri...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EXPLORATORY — NOT VALIDATED FOR FARMER-FACING RISK WARNINGS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[2013, 2024]","{""high_concern"": ""Current yield < 70% of previous-three-obser..."



Evidence comparison saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\model_evidence_comparison.csv


In [4]:

import json
from pathlib import Path
import pandas as pd

REGISTRY_DIR = MODELS_DIR / "registry"
REGISTRY_DIR.mkdir(parents=True, exist_ok=True)

def read_json(path):
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

yield_metadata = read_json(
    MODELS_DIR / "yield_prediction"
    / "individual_crop_hgb_location_metadata.json"
)
yield_readiness = read_json(
    MODELS_DIR / "yield_prediction"
    / "yield_model_deployment_readiness_report.json"
)
low_yield_evaluation = read_json(
    MODELS_DIR / "weather_intelligence"
    / "low_yield_model_evaluation.json"
)
weather_metadata = read_json(
    MODELS_DIR / "weather_intelligence"
    / "weather_intelligence_metadata.json"
)
risk_metadata = read_json(
    MODELS_DIR / "agricultural_risk_intelligence"
    / "agricultural_risk_metadata.json"
)
crop_recommendation_metadata = read_json(
    MODELS_DIR / "crop_recommendation"
    / "crop_recommendation_metadata.json"
)
crop_recommendation_evaluation = read_json(
    MODELS_DIR / "crop_recommendation"
    / "crop_recommendation_evaluation.json"
)

registry_records = [
    {
        "registry_id": "yield_prediction_individual_crop_hgb",
        "component_type": "trained_model",
        "artifact_path": (
            "models/yield_prediction/"
            "individual_crop_hgb_location.joblib"
        ),
        "metadata_path": (
            "models/yield_prediction/"
            "individual_crop_hgb_location_metadata.json"
        ),
        "task": "Yield regression",
        "target": (yield_metadata or {}).get("target"),
        "model_name": (yield_metadata or {}).get("model_name"),
        "recorded_status": (yield_readiness or {}).get("verdict", "UNKNOWN"),
        "registry_disposition": "CONDITIONAL — further validation required",
        "evidence_status": "Metadata and readiness report available",
        "restrictions": (
            "Not production-approved; validate required inputs, "
            "unseen-category reliability, and operational behavior."
        )
    },
    {
        "registry_id": "weather_low_yield_classifier",
        "component_type": "experimental_model_evaluation",
        "artifact_path": None,
        "metadata_path": (
            "models/weather_intelligence/"
            "weather_intelligence_metadata.json"
        ),
        "task": (low_yield_evaluation or {}).get("model_task"),
        "target": (low_yield_evaluation or {}).get("target_definition"),
        "model_name": None,
        "recorded_status": (
            (low_yield_evaluation or {}).get("deployment_status", "UNKNOWN")
        ),
        "registry_disposition": "NOT READY",
        "evidence_status": "Evaluation report available; model artifact not verified",
        "restrictions": (
            "Do not deploy or use as an operational low-yield warning "
            "without inspecting the evaluation evidence and confirming "
            "the corresponding model artifact."
        )
    },
    {
        "registry_id": "agricultural_risk_indicators",
        "component_type": "analytical_output",
        "artifact_path": (
            "models/agricultural_risk_intelligence/"
            "yield_risk_indicators.csv"
        ),
        "metadata_path": (
            "models/agricultural_risk_intelligence/"
            "agricultural_risk_metadata.json"
        ),
        "task": "Exploratory historical risk indicators",
        "target": None,
        "model_name": None,
        "recorded_status": (
            (risk_metadata or {}).get("deployment_status", "UNKNOWN")
        ),
        "registry_disposition": "EXPLORATORY — NOT VALIDATED",
        "evidence_status": "Analytical output and metadata available",
        "restrictions": (
            "Not validated for farmer-facing risk warnings; "
            "do not present as calibrated future risk."
        )
    },
    {
        "registry_id": "crop_recommendation",
        "component_type": "recommendation_output_and_metadata",
        "artifact_path": (
            "models/crop_recommendation/crop_recommendations_ranked.csv"
        ),
        "metadata_path": (
            "models/crop_recommendation/crop_recommendation_metadata.json"
        ),
        "task": "Crop recommendation",
        "target": None,
        "model_name": (crop_recommendation_metadata or {}).get("model_name"),
        "recorded_status": "UNKNOWN — inspect full evaluation evidence",
        "registry_disposition": "UNVERIFIED",
        "evidence_status": (
            "Metadata and evaluation JSON exist; metrics and approval "
            "status have not yet been reviewed."
        ),
        "restrictions": (
            "Do not infer recommendation validity or deployment readiness "
            "from the existence of ranked outputs."
        )
    }
]

registry_df = pd.DataFrame(registry_records)

# Verify the referenced paths rather than trusting filenames alone.
def path_exists(relative_path):
    if not relative_path:
        return False
    return (PROJECT_ROOT / relative_path).exists()

registry_df["artifact_exists"] = registry_df["artifact_path"].apply(path_exists)
registry_df["metadata_exists"] = registry_df["metadata_path"].apply(path_exists)

registry_path = REGISTRY_DIR / "model_registry.csv"
registry_df.to_csv(registry_path, index=False)

print("Registry entries:", len(registry_df))
display(registry_df)
print("\nRegistry saved:", registry_path)


Registry entries: 4


,registry_id,component_type,artifact_path,metadata_path,task,target,model_name,recorded_status,registry_disposition,evidence_status,restrictions,artifact_exists,metadata_exists
0,yield_prediction_individual_crop_hgb,trained_model,models/yield_prediction/individual_crop_hgb_lo...,models/yield_prediction/individual_crop_hgb_lo...,Yield regression,Yield,HistGradientBoostingRegressor,CONDITIONALLY READY FOR INTEGRATION,CONDITIONAL — further validation required,Metadata and readiness report available,Not production-approved; validate required inp...,True,True
1,weather_low_yield_classifier,experimental_model_evaluation,None,models/weather_intelligence/weather_intelligen...,Experimental low-yield classification,Current yield below 80% of the median of the p...,None,NOT READY,NOT READY,Evaluation report available; model artifact no...,Do not deploy or use as an operational low-yie...,False,True
2,agricultural_risk_indicators,analytical_output,models/agricultural_risk_intelligence/yield_ri...,models/agricultural_risk_intelligence/agricult...,Exploratory historical risk indicators,None,None,EXPLORATORY — NOT VALIDATED FOR FARMER-FACING ...,EXPLORATORY — NOT VALIDATED,Analytical output and metadata available,Not validated for farmer-facing risk warnings;...,True,True
3,crop_recommendation,recommendation_output_and_metadata,models/crop_recommendation/crop_recommendation...,models/crop_recommendation/crop_recommendation...,Crop recommendation,None,HistGradientBoostingRegressor,UNKNOWN — inspect full evaluation evidence,UNVERIFIED,Metadata and evaluation JSON exist; metrics an...,Do not infer recommendation validity or deploy...,True,True



Registry saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\registry\model_registry.csv


In [5]:

policy_rules = [
    {
        "policy_id": "POL-001",
        "scope": "all_components",
        "rule": "Verify artifact and metadata paths before use.",
        "severity": "BLOCKING",
        "condition": "Required artifact or metadata is missing.",
        "action": "Block model invocation and record the missing dependency."
    },
    {
        "policy_id": "POL-002",
        "scope": "yield_prediction_individual_crop_hgb",
        "rule": "Require valid mandatory input fields.",
        "severity": "BLOCKING",
        "condition": "Required crop, year, or other contract field is missing or malformed.",
        "action": "Reject the request before prediction."
    },
    {
        "policy_id": "POL-003",
        "scope": "yield_prediction_individual_crop_hgb",
        "rule": "Treat unseen categories as unverified.",
        "severity": "HIGH",
        "condition": "Crop, season, state, or district was not represented in validated evaluation evidence.",
        "action": "Flag the prediction for caution or block it according to application risk policy."
    },
    {
        "policy_id": "POL-004",
        "scope": "weather_low_yield_classifier",
        "rule": "Do not deploy a model marked NOT READY.",
        "severity": "BLOCKING",
        "condition": "Recorded deployment status is NOT READY or model artifact is unavailable.",
        "action": "Prevent operational use until evidence and artifact checks pass."
    },
    {
        "policy_id": "POL-005",
        "scope": "agricultural_risk_indicators",
        "rule": "Do not present exploratory indicators as calibrated future risk.",
        "severity": "BLOCKING",
        "condition": "Risk output lacks validated calibration and documented predictive performance.",
        "action": "Restrict output to exploratory analysis and clearly label its limitations."
    },
    {
        "policy_id": "POL-006",
        "scope": "crop_recommendation",
        "rule": "Verify recommendation model identity and evaluation evidence.",
        "severity": "BLOCKING",
        "condition": "Model artifact identity, target contract, or evaluation status is unresolved.",
        "action": "Keep recommendation component unverified until the metadata and evaluation are reconciled."
    },
    {
        "policy_id": "POL-007",
        "scope": "all_predictive_components",
        "rule": "Preserve evaluation context.",
        "severity": "HIGH",
        "condition": "Metrics are displayed without their evaluation period, sample size, or known limitations.",
        "action": "Attach evaluation context and limitations to reports and downstream interfaces."
    },
    {
        "policy_id": "POL-008",
        "scope": "all_components",
        "rule": "Do not equate registry presence with production approval.",
        "severity": "BLOCKING",
        "condition": "A component is registered but production-readiness evidence is incomplete.",
        "action": "Keep deployment approval separate from registry registration."
    }
]

policy_df = pd.DataFrame(policy_rules)

policy_path = REGISTRY_DIR / "model_policy_rules.json"
with open(policy_path, "w", encoding="utf-8") as f:
    json.dump(policy_rules, f, indent=2)

print("Policy rules created:", len(policy_df))
display(policy_df)

print("\nPolicy rules saved:", policy_path)


Policy rules created: 8


,policy_id,scope,rule,severity,condition,action
0,POL-001,all_components,Verify artifact and metadata paths before use.,BLOCKING,Required artifact or metadata is missing.,Block model invocation and record the missing ...
1,POL-002,yield_prediction_individual_crop_hgb,Require valid mandatory input fields.,BLOCKING,"Required crop, year, or other contract field i...",Reject the request before prediction.
2,POL-003,yield_prediction_individual_crop_hgb,Treat unseen categories as unverified.,HIGH,"Crop, season, state, or district was not repre...",Flag the prediction for caution or block it ac...
3,POL-004,weather_low_yield_classifier,Do not deploy a model marked NOT READY.,BLOCKING,Recorded deployment status is NOT READY or mod...,Prevent operational use until evidence and art...
4,POL-005,agricultural_risk_indicators,Do not present exploratory indicators as calib...,BLOCKING,Risk output lacks validated calibration and do...,Restrict output to exploratory analysis and cl...
5,POL-006,crop_recommendation,Verify recommendation model identity and evalu...,BLOCKING,"Model artifact identity, target contract, or e...",Keep recommendation component unverified until...
6,POL-007,all_predictive_components,Preserve evaluation context.,HIGH,Metrics are displayed without their evaluation...,Attach evaluation context and limitations to r...
7,POL-008,all_components,Do not equate registry presence with productio...,BLOCKING,A component is registered but production-readi...,Keep deployment approval separate from registr...



Policy rules saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\registry\model_policy_rules.json


In [6]:

from pathlib import Path
import json

crop_rec_dir = MODELS_DIR / "crop_recommendation"

crop_rec_metadata_path = (
    crop_rec_dir / "crop_recommendation_metadata.json"
)
crop_rec_evaluation_path = (
    crop_rec_dir / "crop_recommendation_evaluation.json"
)

for path in [crop_rec_metadata_path, crop_rec_evaluation_path]:
    print("\n" + "=" * 80)
    print("File:", path.relative_to(PROJECT_ROOT))
    print("Exists:", path.exists())
    print("=" * 80)

    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            content = json.load(f)

        print(json.dumps(content, indent=2, default=str))

        if path == crop_rec_metadata_path:
            referenced_artifact = content.get("model_artifact")

            if referenced_artifact:
                artifact_path = PROJECT_ROOT / referenced_artifact
                print("\nReferenced artifact:", referenced_artifact)
                print("Referenced artifact exists:", artifact_path.exists())
            else:
                print("\nNo model_artifact field recorded in metadata.")

print("\nAudit complete. No files were modified.")



File: models\crop_recommendation\crop_recommendation_metadata.json
Exists: True
{
  "project": "BharatCrop Intelligence",
  "model_name": "HistGradientBoostingRegressor",
  "model_artifact": "models\\yield_prediction\\individual_crop_hgb_location.joblib",
  "target": "Yield",
  "recommendation_method": "Rank eligible historical crop candidates by predicted yield within State-District-Season-Year groups.",
  "prediction_year_configured": 2025,
  "evaluation_years": [
    2023,
    2024
  ],
  "features": [
    "Crop",
    "Season",
    "Crop_Category",
    "State",
    "District",
    "FE_YearStart"
  ],
  "limitations": [
    "The saved ranking CSV contains retrospective rankings for 2023\u20132024.",
    "Only historically observed and eligible crop candidates are included.",
    "Predicted yield does not establish profitability or agronomic suitability.",
    "Soil pH, NPK, soil moisture, and irrigation features are unavailable.",
    "Weather mapping and full-season alignment are n

In [7]:

import json
import pandas as pd
from pathlib import Path

registry_path = REGISTRY_DIR / "model_registry.csv"

registry_df = pd.read_csv(registry_path)

with open(crop_rec_metadata_path, "r", encoding="utf-8") as f:
    crop_rec_metadata = json.load(f)

with open(crop_rec_evaluation_path, "r", encoding="utf-8") as f:
    crop_rec_evaluation = json.load(f)

crop_rec_mask = registry_df["registry_id"].eq("crop_recommendation")

if crop_rec_mask.sum() != 1:
    raise ValueError(
        "Expected exactly one crop_recommendation registry entry."
    )

idx = registry_df.index[crop_rec_mask][0]

# The recommendation layer reuses the yield model; it is not a
# separate trained estimator according to the inspected metadata.
registry_df.loc[idx, "component_type"] = "derived_ranking_component"
registry_df.loc[idx, "task"] = (
    "Retrospective ranking of eligible historical crop candidates "
    "by predicted yield"
)
registry_df.loc[idx, "target"] = "Predicted Yield used for ranking"
registry_df.loc[idx, "model_name"] = crop_rec_metadata.get("model_name")
registry_df.loc[idx, "recorded_status"] = (
    "EVALUATED RETROSPECTIVELY — NOT PRODUCTION-APPROVED"
)
registry_df.loc[idx, "registry_disposition"] = (
    "RESEARCH / HISTORICAL RANKING ONLY"
)
registry_df.loc[idx, "evidence_status"] = (
    "Metadata and evaluation report available; ranking uses the "
    "existing yield regression artifact."
)
registry_df.loc[idx, "restrictions"] = (
    "Not a separate trained model. Rankings do not establish "
    "profitability or agronomic suitability. The 2025 configured "
    "prediction year is beyond the 2013–2022 training period. "
    "Do not approve for production without further validation."
)
registry_df.loc[idx, "artifact_exists"] = (
    (PROJECT_ROOT / crop_rec_metadata["model_artifact"]).exists()
)
registry_df.loc[idx, "metadata_exists"] = crop_rec_metadata_path.exists()

registry_df.to_csv(registry_path, index=False)

ranking_evidence = {
    "registry_id": "crop_recommendation",
    "evaluation_period": crop_rec_evaluation.get("evaluation_period"),
    "eligible_prediction_rows": crop_rec_evaluation.get(
        "eligible_prediction_rows"
    ),
    "evaluated_groups": crop_rec_evaluation.get("evaluated_groups"),
    "top1_hit_rate": crop_rec_evaluation.get("top1_hit_rate"),
    "top3_hit_rate": crop_rec_evaluation.get("top3_hit_rate"),
    "mean_spearman_correlation": crop_rec_evaluation.get(
        "mean_spearman_correlation"
    ),
    "interpretation": crop_rec_evaluation.get("interpretation"),
    "production_approval": "NOT ESTABLISHED"
}

evidence_path = REGISTRY_DIR / "crop_recommendation_evaluation_evidence.json"

with open(evidence_path, "w", encoding="utf-8") as f:
    json.dump(ranking_evidence, f, indent=2)

print("Registry updated:", registry_path)
print("Ranking evidence saved:", evidence_path)
display(
    registry_df.loc[
        registry_df["registry_id"].eq("crop_recommendation")
    ].T
)
print("\nRecorded evaluation evidence:")
print(json.dumps(ranking_evidence, indent=2))


Registry updated: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\registry\model_registry.csv
Ranking evidence saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\registry\crop_recommendation_evaluation_evidence.json


,3
registry_id,crop_recommendation
component_type,derived_ranking_component
artifact_path,models/crop_recommendation/crop_recommendation...
metadata_path,models/crop_recommendation/crop_recommendation...
task,Retrospective ranking of eligible historical c...
target,Predicted Yield used for ranking
model_name,HistGradientBoostingRegressor
recorded_status,EVALUATED RETROSPECTIVELY — NOT PRODUCTION-APP...
registry_disposition,RESEARCH / HISTORICAL RANKING ONLY
evidence_status,Metadata and evaluation report available; rank...



Recorded evaluation evidence:
{
  "registry_id": "crop_recommendation",
  "evaluation_period": [
    2023,
    2024
  ],
  "eligible_prediction_rows": 8433,
  "evaluated_groups": 2365,
  "top1_hit_rate": 0.7919661733615222,
  "top3_hit_rate": 0.99830866807611,
  "mean_spearman_correlation": 0.7919884758726925,
  "interpretation": "Retrospective ranking evaluation against observed yield among eligible historical crop candidates; not a measure of agronomic suitability or profitability.",
  "production_approval": "NOT ESTABLISHED"
}


In [8]:

import json
import pandas as pd

registry_path = REGISTRY_DIR / "model_registry.csv"
policy_path = REGISTRY_DIR / "model_policy_rules.json"
ranking_evidence_path = (
    REGISTRY_DIR / "crop_recommendation_evaluation_evidence.json"
)

registry_df = pd.read_csv(registry_path)

with open(policy_path, "r", encoding="utf-8") as f:
    policy_rules = json.load(f)

with open(ranking_evidence_path, "r", encoding="utf-8") as f:
    ranking_evidence = json.load(f)

status_summary = []

for _, row in registry_df.iterrows():
    status_summary.append({
        "registry_id": row["registry_id"],
        "component_type": row["component_type"],
        "artifact_exists": bool(row["artifact_exists"]),
        "metadata_exists": bool(row["metadata_exists"]),
        "recorded_status": row["recorded_status"],
        "registry_disposition": row["registry_disposition"],
        "evidence_status": row["evidence_status"],
        "production_approval": "NOT ESTABLISHED",
        "restrictions": row["restrictions"]
    })

status_df = pd.DataFrame(status_summary)

status_csv_path = REGISTRY_DIR / "model_validation_status.csv"
status_json_path = REGISTRY_DIR / "model_validation_status_summary.json"

status_df.to_csv(status_csv_path, index=False)

summary = {
    "project": "BharatCrop Intelligence",
    "registry_entry_count": len(registry_df),
    "policy_rule_count": len(policy_rules),
    "all_registry_entries_have_artifact": bool(
        status_df["artifact_exists"].all()
    ),
    "all_registry_entries_have_metadata": bool(
        status_df["metadata_exists"].all()
    ),
    "production_approval": "NOT ESTABLISHED",
    "components": status_summary,
    "crop_recommendation_evaluation": ranking_evidence,
    "governance_note": (
        "This report records artifact availability and documented evidence. "
        "It does not independently validate model performance or authorize "
        "production deployment."
    )
}

with open(status_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

print("Registry entries:", len(registry_df))
print("Policy rules:", len(policy_rules))
print("All referenced artifacts exist:", status_df["artifact_exists"].all())
print("All referenced metadata exists:", status_df["metadata_exists"].all())
print("\nValidation status by component:")
display(
    status_df[
        [
            "registry_id",
            "recorded_status",
            "registry_disposition",
            "artifact_exists",
            "metadata_exists"
        ]
    ]
)
print("\nCSV saved:", status_csv_path)
print("JSON saved:", status_json_path)
print("\nProduction approval: NOT ESTABLISHED")


Registry entries: 4
Policy rules: 8
All referenced artifacts exist: False
All referenced metadata exists: True

Validation status by component:


,registry_id,recorded_status,registry_disposition,artifact_exists,metadata_exists
0,yield_prediction_individual_crop_hgb,CONDITIONALLY READY FOR INTEGRATION,CONDITIONAL — further validation required,True,True
1,weather_low_yield_classifier,NOT READY,NOT READY,False,True
2,agricultural_risk_indicators,EXPLORATORY — NOT VALIDATED FOR FARMER-FACING ...,EXPLORATORY — NOT VALIDATED,True,True
3,crop_recommendation,EVALUATED RETROSPECTIVELY — NOT PRODUCTION-APP...,RESEARCH / HISTORICAL RANKING ONLY,True,True



CSV saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\registry\model_validation_status.csv
JSON saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\registry\model_validation_status_summary.json

Production approval: NOT ESTABLISHED


# Conclusion: Model Registry and Policy Outputs

## 1. Objective Achieved

This notebook established a traceable registry of four BharatCrop Intelligence components, documented governance policies, and consolidated the available artifact and validation evidence. The registry distinguishes trained models, derived ranking logic, and exploratory analytical outputs rather than treating every output as an independently trained model.

## 2. Registry Findings

- **Yield prediction:** The existing `HistGradientBoostingRegressor` artifact and metadata are available. Its recorded status is *Conditionally Ready for Integration*. Further input-contract, unseen-category, and operational validation is required; this status does not constitute production approval.
- **Crop recommendation:** The recommendation component reuses the yield regression model to rank eligible historical crop candidates. Its retrospective evaluation covers 2023–2024, with a Top-1 hit rate of approximately 79.20%, a Top-3 hit rate of approximately 99.83%, and a mean Spearman correlation of approximately 0.792. These metrics describe historical ranking performance, not profitability or agronomic suitability.
- **Weather-based low-yield classification:** The evaluation report records the component as *NOT READY*. A corresponding model artifact has not been verified at the registered path, so operational use remains restricted.
- **Agricultural risk indicators:** The existing analytical outputs are classified as exploratory and are not validated for farmer-facing risk warnings.

## 3. Governance Outputs

The notebook generated the following files under `models/registry/`:

- `model_registry.csv` — component identities, artifact paths, recorded statuses, restrictions, and evidence availability.
- `model_policy_rules.json` — eight governance rules covering artifact checks, required inputs, unseen categories, evaluation context, and deployment restrictions.
- `crop_recommendation_evaluation_evidence.json` — recorded retrospective ranking metrics and their interpretation.
- `model_validation_status.csv` — a tabular status summary.
- `model_validation_status_summary.json` — a machine-readable consolidated report.

## 4. Limitations and Outstanding Work

The artifact inventory verifies file presence, not model correctness or performance. The missing weather-classifier artifact remains unresolved. Further work is needed to verify model contracts, enforce required-input checks, investigate unseen-category behavior, confirm evaluation procedures, and test application-level reliability and monitoring. Historical ranking results must not be interpreted as evidence of causal effects, profitability, or agronomic suitability.

## 5. Final Assessment

The registry and policy artifacts provide a reproducible starting point for model governance and downstream integration decisions. They do not constitute independent model validation, regulatory compliance, or production approval. Deployment decisions must remain conditional on documented evidence and the completion of outstanding checks.

**Overall status: Registry and policy documentation completed; production approval not established.**